# Residual Multi-Head Mixing GAT on Cora — Demo

End-to-end demo: dataset stats -> baselines (MLP/GCN) -> architecture comparison (GCN/SAGE/GAT) -> proposed model ablation.

**How to run:** open in Colab and `Runtime > Run all`. Everything downloads and runs on CPU or GPU automatically.

Repo: https://github.com/minhdo2207/GraphMl

## 1. Setup — install dependencies & clone the repo

Run this once. On Colab it installs PyTorch Geometric and clones the project so `src/` is importable.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/minhdo2207/GraphMl.git'
REPO_DIR = '/content/GraphMl' if IN_COLAB else os.path.abspath('..')

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL $REPO_DIR
    # torch is preinstalled on Colab; add PyG matching the installed torch/CUDA build
    !pip -q install torch-geometric

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo dir:', REPO_DIR)

In [ ]:
import torch, torch_geometric
print('torch', torch.__version__, '| pyg', torch_geometric.__version__,
      '| device', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. Dataset statistics (Person 2)

Loads Cora from PyTorch Geometric and prints node/edge/feature/class counts and the train/val/test mask sanity check.

In [ ]:
from src.data import get_data, print_statistics

data, num_features, num_classes = get_data(name='Cora')
_ = print_statistics(data, num_features, num_classes, name='Cora')

## 3. Configuration

All hyperparameters live in one `Config`. Change fields here (e.g. `seeds=3` for a faster demo).

In [ ]:
from src.utils import load_config

cfg = load_config('configs/default.yaml', seeds=3)   # 3 seeds keeps the demo quick
cfg.to_dict()

## 4. Baselines — MLP vs GCN (Person 2)

Does the graph help? Compare a feature-only MLP against a 2-layer GCN.

In [ ]:
from src.training import run_seeds
from src.utils.plots import print_table

baseline_rows, baseline_runs = [], {}
for name in ['mlp', 'gcn']:
    s = run_seeds(cfg, model_name=name, verbose=True)
    baseline_rows.append(s.as_row())
    baseline_runs[name] = s.runs[0]

print_table(baseline_rows)

## 5. Architecture comparison — GCN / GraphSAGE / GAT (Person 3)

In [ ]:
comp_rows, comp_runs = [], {}
for name in ['gcn', 'graphsage', 'gat']:
    s = run_seeds(cfg, model_name=name, verbose=True)
    comp_rows.append(s.as_row())
    comp_runs[name] = s.runs[0]

print_table(comp_rows)

## 6. Proposed model + ablation (Person 4)

Residual Multi-Head Mixing GAT. Ablate the residual skip and DropEdge independently.

In [ ]:
variants = {
    'no residual, no dropedge': cfg.update(residual=False, drop_edge=0.0),
    '+ residual':               cfg.update(residual=True,  drop_edge=0.0),
    '+ residual + dropedge':    cfg.update(residual=True,  drop_edge=0.2),
}

ablation_rows, ablation_runs = [], {}
for label, vcfg in variants.items():
    s = run_seeds(vcfg, model_name='proposed', verbose=True)
    row = s.as_row(); row['variant'] = label
    ablation_rows.append(row)
    ablation_runs[label] = s.runs[0]

print_table(ablation_rows)

## 7. Training curves (Person 5)

Plot validation-accuracy / training-loss curves and show them inline.

In [ ]:
from src.utils.plots import plot_curves
from IPython.display import Image, display

all_runs = {**comp_runs, 'proposed': ablation_runs['+ residual + dropedge']}
path = plot_curves(all_runs, 'figures/demo_curves.png', title='Cora — model comparison')
display(Image(path))

## 8. Save result tables to CSV

CSVs land in `results/` — paste them into the LaTeX/Word report.

In [ ]:
from src.utils.plots import save_results_table

save_results_table(baseline_rows, 'results/baselines.csv')
save_results_table(comp_rows,     'results/comparison.csv')
save_results_table(ablation_rows, 'results/ablation.csv')
print('Saved results/baselines.csv, results/comparison.csv, results/ablation.csv')